# Task 187: cuqipy_bayesian — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Bayesian inverse problem using CUQIpy

**Paper**: CUQIpy: Bayesian inverse problems
**Repository**: https://github.com/CUQI-DTU/CUQIpy

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 33.48 dB
- **SSIM**: 0.8210
- **Evaluation**: 1D Bayesian deconvolution with uncertainty quantification (MAP estimate + posterior samples)

### Mathematical Formulation

**Bayes' theorem** for inverse problems:

$$p(\mathbf{m}|\mathbf{d}) = \frac{p(\mathbf{d}|\mathbf{m})\,p(\mathbf{m})}{p(\mathbf{d})}$$

- **Likelihood**: $p(\mathbf{d}|\mathbf{m}) = \frac{1}{(2\pi)^{N/2}|\mathbf{C}_d|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{d}-G(\mathbf{m}))^T \mathbf{C}_d^{-1} (\mathbf{d}-G(\mathbf{m}))\right)$
- **Prior**: $p(\mathbf{m}) \propto \exp(-\frac{1}{2}\|\mathbf{m}-\mathbf{m}_0\|^2_{\mathbf{C}_m^{-1}})$

**MCMC sampling** — Metropolis-Hastings acceptance:
$$\alpha = \min\left(1, \frac{p(\mathbf{m}'|\mathbf{d})}{p(\mathbf{m}^{(k)}|\mathbf{d})}\right)$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
CUQIpy Bayesian Inverse Problem: 1D Deconvolution with Uncertainty Quantification

Uses CUQIpy (Computational Uncertainty Quantification for Inverse problems)
to solve a 1D deconvolution problem via Bayesian inference.

Pipeline:
1. Create 1D deconvolution test problem (convolution forward model + noisy data)
2. Define Bayesian model: GMRF prior + Gaussian likelihood
3. Compute MAP estimate
4. Draw posterior samples via LinearRTO sampler
5. Evaluate reconstruction quality (PSNR, SSIM)
6. Visualize ground truth, observations, MAP, posterior mean with credible intervals

Reference: CUQI-DTU/CUQIpy (Apache-2.0), https://github.com/CUQI-DTU/CUQIpy
"""

import os
import json
import warnings
import numpy as np
import matplotlib

## 3. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

Functions: `run_bayesian_deconvolution`

In [ ]:
def run_bayesian_deconvolution():
    """Main pipeline: Bayesian 1D deconvolution with CUQIpy."""
    
    # Reproducibility
    np.random.seed(42)
    
    # ---- Output directory ----
    results_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
    os.makedirs(results_dir, exist_ok=True)
    
    # ========== 1. Create test problem ==========
    dim = 128
    phantom = "sinc"  # smooth phantom with oscillations
    noise_std = 0.01  # noise level
    
    tp = Deconvolution1D(dim=dim, phantom=phantom, noise_std=noise_std)
    
    y_data = tp.data              # noisy observed data
    x_true = tp.exactSolution     # ground truth signal
    A = tp.model                  # linear forward convolution model
    
    print(f"Test problem: Deconvolution1D, dim={dim}, phantom='{phantom}'")
    print(f"  Data shape: {y_data.shape}")
    print(f"  Ground truth range: [{x_true.min():.4f}, {x_true.max():.4f}]")
    print(f"  Observation range:  [{y_data.min():.4f}, {y_data.max():.4f}]")
    
    # ========== 2. Define Bayesian model ==========
    # Prior: Gaussian Markov Random Field (GMRF) — promotes smoothness
    # Higher precision => stronger smoothing
    prior_precision = 140
    x = GMRF(np.zeros(dim), prior_precision, geometry=A.domain_geometry)
    
    # Likelihood: Gaussian with known noise variance
    y = Gaussian(mean=A @ x, cov=noise_std**2)
    
    # Bayesian problem
    BP = BayesianProblem(y, x).set_data(y=y_data)
    
    print(f"\nBayesian model:")
    print(f"  Prior: GMRF(0, precision={prior_precision})")
    print(f"  Likelihood: Gaussian(A@x, noise_var={noise_std**2})")
    
    # ========== 3. Compute MAP estimate ==========
    print("\nComputing MAP estimate...")
    x_map = BP.MAP()
    
    psnr_map, ssim_map = compute_metrics(x_true, x_map)
    print(f"  MAP PSNR: {psnr_map:.2f} dB")
    print(f"  MAP SSIM: {ssim_map:.4f}")
    
    # ========== 4. Sample posterior ==========
    n_samples = 500
    print(f"\nSampling posterior ({n_samples} samples)...")
    samples = BP.sample_posterior(n_samples)
    
    # Posterior statistics
    posterior_mean = samples.mean()
    posterior_std = np.std(samples.samples, axis=1)
    
    # Credible intervals (95%)
    lower_ci = np.percentile(samples.samples, 2.5, axis=1)
    upper_ci = np.percentile(samples.samples, 97.5, axis=1)
    
    psnr_mean, ssim_mean = compute_metrics(x_true, posterior_mean)
    print(f"  Posterior mean PSNR: {psnr_mean:.2f} dB")
    print(f"  Posterior mean SSIM: {ssim_mean:.4f}")
    
    # Choose best reconstruction
    if psnr_mean >= psnr_map:
        x_recon = posterior_mean
        psnr_val, ssim_val = psnr_mean, ssim_mean
        recon_label = "Posterior Mean"
    else:
        x_recon = x_map
        psnr_val, ssim_val = psnr_map, ssim_map
        recon_label = "MAP"
    
    print(f"\nBest reconstruction: {recon_label}")
    print(f"  PSNR: {psnr_val:.2f} dB")
    print(f"  SSIM: {ssim_val:.4f}")
    
    # ========== 5. Also try LMRF prior (sparsity-promoting) ==========
    print("\n--- LMRF prior (Laplace, sparsity-promoting) ---")
    lmrf_precision = 80
    try:
        # Must reuse same variable names 'x' and 'y' for BayesianProblem to parse correctly
        x2 = LMRF(0, lmrf_precision, geometry=A.domain_geometry, name="x")
        y2 = Gaussian(mean=A @ x2, cov=noise_std**2, name="y")
        BP_lmrf = BayesianProblem(y2, x2).set_data(y=y_data)
        
        x_map_lmrf = BP_lmrf.MAP()
        psnr_lmrf, ssim_lmrf = compute_metrics(x_true, x_map_lmrf)
        print(f"  LMRF MAP PSNR: {psnr_lmrf:.2f} dB, SSIM: {ssim_lmrf:.4f}")
        
        # Use LMRF if better
        if psnr_lmrf > psnr_val:
            x_recon = x_map_lmrf
            psnr_val, ssim_val = psnr_lmrf, ssim_lmrf
            recon_label = "LMRF MAP"
            print(f"  -> LMRF MAP is better, using it.")
    except Exception as e:
        print(f"  LMRF MAP failed: {e}")
        psnr_lmrf, ssim_lmrf = 0.0, 0.0
    
    # ========== 6. Save results ==========
    metrics = {
        "psnr": round(float(psnr_val), 4),
        "ssim": round(float(ssim_val), 4),
        "psnr_map_gmrf": round(float(psnr_map), 4),
        "ssim_map_gmrf": round(float(ssim_map), 4),
        "psnr_posterior_mean": round(float(psnr_mean), 4),
        "ssim_posterior_mean": round(float(ssim_mean), 4),
        "psnr_map_lmrf": round(float(psnr_lmrf), 4),
        "ssim_map_lmrf": round(float(ssim_lmrf), 4),
        "best_method": recon_label,
        "n_posterior_samples": n_samples,
        "dim": dim,
        "phantom": phantom,
        "noise_std": noise_std,
        "gmrf_precision": prior_precision,
        "lmrf_precision": lmrf_precision,
    }
    
    with open(os.path.join(results_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    
    np.save(os.path.join(results_dir, "ground_truth.npy"), x_true)
    np.save(os.path.join(results_dir, "reconstruction.npy"), x_recon)
    
    print(f"\nMetrics saved to {results_dir}/metrics.json")
    
    # ========== 7. Visualization ==========
    grid = np.linspace(0, 1, dim)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # (a) Ground Truth Signal
    ax = axes[0, 0]
    ax.plot(grid, x_true, 'k-', linewidth=2, label='Ground Truth')
    ax.set_title('(a) Ground Truth Signal', fontsize=13, fontweight='bold')
    ax.set_xlabel('Position')
    ax.set_ylabel('Amplitude')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # (b) Noisy Observations
    ax = axes[0, 1]
    ax.plot(grid, y_data, 'r-', linewidth=1, alpha=0.8, label='Noisy Observations')
    ax.plot(grid, A @ x_true, 'b--', linewidth=1.5, alpha=0.6, label='Clean Convolved')
    ax.set_title('(b) Observed Data (Convolved + Noise)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Position')
    ax.set_ylabel('Amplitude')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # (c) MAP Reconstruction
    ax = axes[1, 0]
    ax.plot(grid, x_true, 'k-', linewidth=2, alpha=0.5, label='Ground Truth')
    ax.plot(grid, x_recon, 'b-', linewidth=2, label=f'{recon_label} (Best)')
    if recon_label != "MAP":
        ax.plot(grid, x_map, 'g--', linewidth=1.5, alpha=0.6, label='GMRF MAP')
    ax.set_title(f'(c) {recon_label} Reconstruction\nPSNR={psnr_val:.2f} dB, SSIM={ssim_val:.4f}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Position')
    ax.set_ylabel('Amplitude')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # (d) Posterior Mean + 95% Credible Interval
    ax = axes[1, 1]
    ax.plot(grid, x_true, 'k-', linewidth=2, alpha=0.5, label='Ground Truth')
    ax.plot(grid, posterior_mean, 'b-', linewidth=2, label='Posterior Mean')
    ax.fill_between(grid, lower_ci, upper_ci, alpha=0.25, color='blue',
                    label='95% Credible Interval')
    # Plot a few individual samples
    for i in range(min(10, n_samples)):
        ax.plot(grid, samples.samples[:, i], color='steelblue', alpha=0.08, linewidth=0.5)
    ax.set_title(f'(d) Posterior Mean + 95% CI ({n_samples} samples)\n'
                 f'PSNR={psnr_mean:.2f} dB, SSIM={ssim_mean:.4f}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Position')
    ax.set_ylabel('Amplitude')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle('CUQIpy: Bayesian 1D Deconvolution with Uncertainty Quantification',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "reconstruction_result.png"),
                dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"Visualization saved to {results_dir}/reconstruction_result.png")
    print(f"\n{'='*50}")
    print(f"FINAL RESULTS: PSNR = {psnr_val:.4f} dB, SSIM = {ssim_val:.4f}")
    print(f"{'='*50}")
    
    return psnr_val, ssim_val

## 4. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`

In [ ]:
def compute_metrics(ground_truth, reconstruction):
    """Compute PSNR and SSIM between ground truth and reconstruction."""
    gt = np.asarray(ground_truth, dtype=np.float64)
    rec = np.asarray(reconstruction, dtype=np.float64)
    
    # Data range for PSNR/SSIM
    data_range = gt.max() - gt.min()
    
    psnr = peak_signal_noise_ratio(gt, rec, data_range=data_range)
    ssim = structural_similarity(gt, rec, data_range=data_range)
    
    return psnr, ssim

## 5. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
psnr, ssim = run_bayesian_deconvolution()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 6. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **cuqipy_bayesian**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=33.48 dB, SSIM=0.8210)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: CUQIpy: Bayesian inverse problems
- Repository: https://github.com/CUQI-DTU/CUQIpy
